# STEP 3. 실험 실행
3개 모델 × 2 조건(no-RAG / RAG) → 응답 저장

In [1]:
# [셀1] 경로 설정 + QA 데이터셋 로드
import json
from pathlib import Path

QA_PATH     = Path(r"D:\충북대\지능화캡스톤\data\qa_dataset.json")
INDEX_DIR   = Path(r"D:\rag_index")
RESULT_PATH = Path(r"D:\충북대\지능화캡스톤\data\experiment_results.json")

with open(QA_PATH, encoding="utf-8") as f:
    qa_dataset = json.load(f)

print(f"QA 데이터셋 로드: {len(qa_dataset)}개")
print("샘플:", qa_dataset[0]["question"])

QA 데이터셋 로드: 83개
샘플: 집 안에서 지진 발생 시 안전한 대피 공간으로 적합한 장소는 어디인가요?


In [2]:
# [셀2] FAISS 인덱스 + 임베딩 모델 로드
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("임베딩 모델 로딩 중...")
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectordb = FAISS.load_local(
    str(INDEX_DIR), embed_model,
    allow_dangerous_deserialization=True
)
print(f"인덱스 로드 완료: {vectordb.index.ntotal}개 벡터")

임베딩 모델 로딩 중...


C:\Users\hyebi\AppData\Local\Temp\ipykernel_62060\1045866062.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(
d:\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32578.57it/s]


인덱스 로드 완료: 305개 벡터


In [3]:
# [셀3] 모델 목록 + 실험 조건 정의
MODELS = [
    {"name": "exaone",  "ollama_model": "exaone3.5:7.8b"},
    {"name": "qwen3",   "ollama_model": "qwen3:8b"},
    {"name": "llama31", "ollama_model": "llama3.1:8b"},
]
CONDITIONS = ["no_rag", "rag"]

print("실험 조건:")
for m in MODELS:
    for c in CONDITIONS:
        print(f"  {m['name']} × {c}")
total = len(MODELS) * len(CONDITIONS) * len(qa_dataset)
print(f"\n총 {total}번 추론 예정")

실험 조건:
  exaone × no_rag
  exaone × rag
  qwen3 × no_rag
  qwen3 × rag
  llama31 × no_rag
  llama31 × rag

총 498번 추론 예정


In [4]:
# [셀4] RAG 컨텍스트 검색 함수
def retrieve_context(question: str, k: int = 3) -> str:
    docs = vectordb.similarity_search(question, k=k)
    return "\n\n".join(d.page_content for d in docs)

# 검색 테스트
test_q = qa_dataset[0]["question"]
ctx = retrieve_context(test_q)
print(f"Q: {test_q}")
print(f"검색된 컨텍스트 ({len(ctx)}자):")
print(ctx[:300], "...")

Q: 집 안에서 지진 발생 시 안전한 대피 공간으로 적합한 장소는 어디인가요?
검색된 컨텍스트 (1031자):
탁자 밑에 들어가 안전한 자세를 취한 후 진동이 멈추면 건물 밖으로 대피합니다. 
내진설계가 되어 있는 건물은 지진에 대한 안전성이 높을 수 있으니, 무리하게 대피하기 
보다는 진동이 멈추면 주변 상황을 살핀 후 대처합니다. 
지진으로 인하여 불이 났을 때에는 가장 먼저 무엇을 해야 하나요?
화재가 발생한 것을 아는 순간 “불이야”라고 외쳐 주변 사람들에게 알리는 것이 가장 
중요합니다. 그 후 서로 도와 초기에 불을 끄는 것이 화재로 인한 2차 피해를 막을 수 있는 
방법입니다. 
Q8 
Q9
Q10
Q11
37 미리 대비하고 알 ...


In [5]:
# [셀5] 프롬프트 + 추론 함수
from langchain_ollama import OllamaLLM
import time

NO_RAG_PROMPT = (
    "당신은 재난안전 전문가입니다.\n"
    "다음 질문에 2~4문장으로 간결하게 답하세요.\n\n"
    "질문: {question}\n답변:"
)

RAG_PROMPT = (
    "당신은 재난안전 전문가입니다.\n"
    "아래 [참고 문서]를 바탕으로 질문에 2~4문장으로 간결하게 답하세요.\n"
    "참고 문서 외의 내용은 추가하지 마세요.\n\n"
    "[참고 문서]\n{context}\n\n"
    "질문: {question}\n답변:"
)


def run_inference(ollama_model: str, question: str, context: str = None) -> dict:
    llm = OllamaLLM(
        model=ollama_model,
        temperature=0.1,
        num_ctx=4096,
    )
    if context:
        prompt = RAG_PROMPT.format(context=context[:1500], question=question)
    else:
        prompt = NO_RAG_PROMPT.format(question=question)

    t0 = time.time()
    answer = llm.invoke(prompt)
    elapsed = round(time.time() - t0, 2)
    return {"answer": answer.strip(), "elapsed_sec": elapsed}


# 연결 테스트
test_res = run_inference("exaone3.5:7.8b", qa_dataset[0]["question"])
print("[테스트]", test_res["answer"][:120])
print(f"소요시간: {test_res['elapsed_sec']}초")


[테스트] 집 안에서 지진 발생 시 가장 안전한 대피 공간은 **내진 설계된 벽체가 있는 방의 가장 안쪽 모서리**입니다. 특히 **벽 근처의 튼튼한 가구 밑**이나 **튼튼한 구조물 아래**는 충격을 완화하는 데 효과적입니다
소요시간: 3.72초


In [6]:
# [셀6] 전체 실험 실행 (약 1~3시간 소요)
# 중단 후 재실행해도 이미 완료된 항목은 건너뜁니다.
from tqdm import tqdm

# 기존 결과 이어받기
if RESULT_PATH.exists():
    with open(RESULT_PATH, encoding="utf-8") as f:
        all_results = json.load(f)
    print(f"기존 결과 {len(all_results)}개 로드 → 이어서 실행")
else:
    all_results = []

done_keys = {
    (r["id"], r["model"], r["condition"])
    for r in all_results
}

for model_cfg in MODELS:
    model_name   = model_cfg["name"]
    ollama_model = model_cfg["ollama_model"]

    for condition in CONDITIONS:
        use_rag = (condition == "rag")
        pbar = tqdm(qa_dataset, desc=f"{model_name}/{condition}", ncols=90)

        for qa in pbar:
            key = (qa["id"], model_name, condition)
            if key in done_keys:
                continue

            try:
                ctx = retrieve_context(qa["question"]) if use_rag else None
                res = run_inference(ollama_model, qa["question"], context=ctx)

                record = {
                    "id":            qa["id"],
                    "model":         model_name,
                    "condition":     condition,
                    "disaster_type": qa["disaster_type"],
                    "q_type":        qa["type"],
                    "question":      qa["question"],
                    "gold_answer":   qa["gold_answer"],
                    "pred_answer":   res["answer"],
                    "context":       ctx or "",
                    "elapsed_sec":   res["elapsed_sec"],
                }
                all_results.append(record)
                done_keys.add(key)

                # 50건마다 중간 저장
                if len(all_results) % 50 == 0:
                    with open(RESULT_PATH, "w", encoding="utf-8") as f:
                        json.dump(all_results, f, ensure_ascii=False, indent=2)

            except Exception as e:
                print(f"\n  [ERROR] {qa['id']}/{model_name}/{condition}: {e}")
                time.sleep(2)

        pbar.close()

# 최종 저장
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n===== 완료: {len(all_results)}개 결과 저장 =====")
print(f"저장 경로: {RESULT_PATH}")

llama31/rag: 100%|████████████████████████████████████████| 83/83 [02:34<00:00,  1.86s/it]


===== 완료: 498개 결과 저장 =====
저장 경로: D:\충북대\지능화캡스톤\data\experiment_results.json


In [7]:
# [셀7] 결과 요약 확인
from collections import Counter

print(f"총 레코드: {len(all_results)}개\n")

print("모델 × 조건별 완료 수:")
key_counts = Counter((r["model"], r["condition"]) for r in all_results)
for (model, cond), cnt in sorted(key_counts.items()):
    print(f"  {model:10s} / {cond:6s}: {cnt}개")

print("\n샘플 결과 (exaone / no_rag):")
sample = next(
    (r for r in all_results if r["model"]=="exaone" and r["condition"]=="no_rag"),
    None
)
if sample:
    print(f"  Q: {sample['question']}")
    print(f"  Gold: {sample['gold_answer'][:120]}")
    print(f"  Pred: {sample['pred_answer'][:120]}")

print("\n다음: 04_evaluation.ipynb")

총 레코드: 498개

모델 × 조건별 완료 수:
  exaone     / no_rag: 83개
  exaone     / rag   : 83개
  llama31    / no_rag: 83개
  llama31    / rag   : 83개
  qwen3      / no_rag: 83개
  qwen3      / rag   : 83개

샘플 결과 (exaone / no_rag):
  Q: 집 안에서 지진 발생 시 안전한 대피 공간으로 적합한 장소는 어디인가요?
  Gold: 탁자 아래와 같이 집 안에서 대피할 수 있는 안전한 대피 공간을 미리 파악해 둡니다.
  Pred: 집 안에서 지진 발생 시 가장 안전한 대피 공간은 **내진 설계된 벽체가 있는 방의 가장 안쪽 모서리**입니다. 특히 **벽 근처의 튼튼한 가구 밑**이나 **튼튼한 테이블 아래**도 안전한 피신처가 될 수 있습니다

다음: 04_evaluation.ipynb
